# SecureFedDrone Complete Pipeline (Modified 26 March)Full implementation including attack, watermarking, detection, and aggregation.

In [ ]:
import torchimport numpy as np

## Byzantine Attack

In [ ]:
def byzantine_attack(grad, beta):    noise = torch.randn_like(grad)    v = grad + beta * noise    return v * (grad.norm() / (v.norm() + 1e-8))

## Watermark Embedding

In [ ]:
def embed_watermark(weights, signature, alpha=0.01):    c = torch.randn_like(weights)    return weights + alpha * c * (2 * signature - 1), c

## Watermark Verification

In [ ]:
def verify_watermark(w, c, signature):    corr = torch.dot(w.flatten(), (c * (2*signature-1)).flatten()) / (w.norm() * c.norm() + 1e-8)    ber = ((signature > 0.5).float().mean())  # simplified    return corr.item(), ber.item()

## Statistical Anomaly Detection (MAD)

In [ ]:
def mad_score(updates):    norms = torch.tensor([u.norm().item() for u in updates])    median = norms.median()    mad = (norms - median).abs().median()    return 0.6745 * (norms - median) / (mad + 1e-8)

## Behavioral Consistency

In [ ]:
def behavior_score(current, history):    cos_sim = torch.dot(current.flatten(), history.flatten()) / (current.norm()*history.norm() + 1e-8)    return max(0, min(1, 1 - cos_sim.item()))

## Fusion Scoring

In [ ]:
def malicious_score(water, stat, behavior):    return 0.5*water + 0.3*stat + 0.2*behavior

## Client Filtering

In [ ]:
def filter_clients(scores):    accepted = [i for i,s in enumerate(scores) if s < 0.5]    rejected = [i for i,s in enumerate(scores) if s > 0.7]    return accepted, rejected

## Simulation Loop

In [ ]:
num_clients = 8betas = [0.3, 0.5, 0.7]for beta in betas:    print(f"Running beta={beta}")    updates = []    histories = []    for i in range(num_clients):        grad = torch.randn(100)        if i in [6,7]:  # malicious            grad = byzantine_attack(grad, beta)        updates.append(grad)        histories.append(torch.randn(100))    stats = mad_score(updates)    scores = []    for i in range(num_clients):        water = np.random.rand()        behavior = behavior_score(updates[i], histories[i])        stat = abs(stats[i].item())        score = malicious_score(water, stat, behavior)        scores.append(score)    accepted, rejected = filter_clients(scores)    print("Accepted:", accepted)    print("Rejected:", rejected)    print("-"*40)